# Notebook 3: Herramientas de análisis

Este notebook explora las herramientas de análisis disponibles en gadget-ng:

1. **FoF (Friends-of-Friends)**: identificación de halos
2. **P(k)**: espectro de potencia de materia
3. **ξ(r)**: función de correlación de 2 puntos
4. **c(M)**: relación concentración-masa
5. **Snapshot metrics**: métricas básicas de una simulación

**Requisitos:** simulaciones del Notebook 2 completadas, numpy, scipy, matplotlib

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

PROJECT_ROOT = Path("..")
BINARY = PROJECT_ROOT / "target" / "release" / "gadget-ng"

# Resultado del Notebook 2
out_cosmo = PROJECT_ROOT / "runs" / "cosmo_nb02"
out_analysis = out_cosmo / "analysis"

assert BINARY.exists(), "Ejecuta primero: cargo build --release -p gadget-ng-cli"
print("Binario disponible:", BINARY.resolve())
print("Directorio de análisis:", out_analysis.resolve())

## 1. Leer el results.json

El subcomando `analyze` produce un `results.json` estructurado con todos los
estadísticos calculados.

In [ ]:
results_file = out_analysis / "results.json"

if not results_file.exists():
    print(f"No se encontró {results_file}")
    print("Ejecuta primero el Notebook 2 para generar los datos.")
    results = {}
else:
    with open(results_file) as f:
        results = json.load(f)

    print("Estructura del results.json:")
    for k, v in results.items():
        if isinstance(v, list):
            print(f"  {k}: lista de {len(v)} elementos")
        elif isinstance(v, dict):
            print(f"  {k}: dict con claves {list(v.keys())[:5]}")
        else:
            print(f"  {k}: {v}")

## 2. Usar snapshot_metrics.py

El script `scripts/analysis/snapshot_metrics.py` calcula métricas básicas
de cualquier snapshot: energía cinética, potencial, cantidad de movimiento, etc.

In [ ]:
metrics_script = PROJECT_ROOT / "scripts" / "analysis" / "snapshot_metrics.py"

if not metrics_script.exists():
    print(f"Script no encontrado: {metrics_script}")
else:
    snap_final = out_cosmo / "snapshot_final"
    if not snap_final.exists():
        snaps = sorted(out_cosmo.glob("snapshot*"))
        snap_final = snaps[-1] if snaps else None

    if snap_final:
        result = subprocess.run(
            [sys.executable, str(metrics_script), str(snap_final)],
            capture_output=True,
            text=True,
        )
        print(result.stdout or "(sin output)")
        if result.returncode != 0:
            print("STDERR:", result.stderr[-500:])
    else:
        print("No se encontró snapshot final. Ejecuta el Notebook 2 primero.")

## 3. Dashboard de análisis completo

Visualización integrada de todos los estadísticos disponibles en `results.json`.

In [ ]:
def safe_array(data: dict, *keys) -> np.ndarray | None:
    """Extrae un array de forma robusta probando múltiples claves."""
    for k in keys:
        v = data.get(k)
        if v is not None:
            arr = np.array(v)
            if arr.size > 0:
                return arr
    return None


fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle("Dashboard de Análisis — gadget-ng", fontsize=15, fontweight="bold")

# ── Panel 1: P(k) ─────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
pk_data = results.get("power_spectrum", results.get("pk", {}))
if isinstance(pk_data, dict):
    k_arr = safe_array(pk_data, "k", "k_h_mpc")
    pk_arr = safe_array(pk_data, "power", "pk_mpc3_h3", "pk")
    if k_arr is not None and pk_arr is not None:
        mask = pk_arr > 0
        ax1.loglog(k_arr[mask], pk_arr[mask], "o-", ms=3, lw=1.5, color="steelblue")
        ax1.set_xlabel(r"$k$ [$h$/Mpc]")
        ax1.set_ylabel(r"$P(k)$ [Mpc/$h$]³")
        ax1.grid(True, alpha=0.3, which="both")
    else:
        ax1.text(0.5, 0.5, "P(k) no disponible", ha="center", va="center",
                 transform=ax1.transAxes, color="gray", fontsize=11)
ax1.set_title("Espectro de potencia P(k)")

# ── Panel 2: ξ(r) ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
xi_data = results.get("correlation_function", results.get("xi", {}))
if isinstance(xi_data, dict):
    r_arr = safe_array(xi_data, "r", "r_mpc_h")
    xi_arr = safe_array(xi_data, "xi", "correlation")
    if r_arr is not None and xi_arr is not None:
        pos_mask = xi_arr > 0
        if pos_mask.any():
            ax2.loglog(r_arr[pos_mask], xi_arr[pos_mask], "o-", ms=3, lw=1.5, color="darkorange")
        ax2.set_xlabel(r"$r$ [Mpc/$h$]")
        ax2.set_ylabel(r"$\xi(r)$")
        ax2.grid(True, alpha=0.3, which="both")
    else:
        ax2.text(0.5, 0.5, "ξ(r) no disponible", ha="center", va="center",
                 transform=ax2.transAxes, color="gray", fontsize=11)
ax2.set_title("Función de correlación ξ(r)")

# ── Panel 3: c(M) ─────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
cm_data = results.get("concentration_mass", results.get("c_m", results.get("cm", {})))
if isinstance(cm_data, dict):
    m_arr = safe_array(cm_data, "mass", "m", "m200")
    c_arr = safe_array(cm_data, "concentration", "c", "c200")
    if m_arr is not None and c_arr is not None:
        ax3.loglog(m_arr, c_arr, "o", ms=4, alpha=0.7, color="green")
        ax3.set_xlabel(r"$M$ [$M_\odot/h$]")
        ax3.set_ylabel(r"$c$")
        ax3.grid(True, alpha=0.3, which="both")
    else:
        ax3.text(0.5, 0.5, "c(M) no disponible", ha="center", va="center",
                 transform=ax3.transAxes, color="gray", fontsize=11)
ax3.set_title("Relación concentración-masa c(M)")

# ── Panel 4: Distribución de masas de halos ────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
halos = results.get("halos", results.get("fof", results.get("groups", [])))
if isinstance(halos, list) and halos:
    masses = []
    for h in halos:
        if isinstance(h, dict):
            m = h.get("mass", h.get("m", h.get("total_mass", 0)))
            if m and m > 0:
                masses.append(m)
    if masses:
        ax4.hist(np.log10(masses), bins=min(20, len(masses) // 2 + 1),
                 edgecolor="black", alpha=0.8, color="mediumpurple")
        ax4.set_xlabel(r"$\log_{10}(M / M_\odot)$")
        ax4.set_ylabel("Número de halos")
        ax4.set_title(f"HMF: {len(masses)} halos FoF")
        ax4.grid(True, alpha=0.3)
    else:
        ax4.text(0.5, 0.5, "Sin masas de halos", ha="center", va="center",
                 transform=ax4.transAxes, color="gray", fontsize=11)
        ax4.set_title("Función de masa de halos")
else:
    ax4.text(0.5, 0.5, "FoF no disponible", ha="center", va="center",
             transform=ax4.transAxes, color="gray", fontsize=11)
    ax4.set_title("Función de masa de halos")

# ── Panel 5: Partículas en el espacio de fase ──────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])

def load_particles_jsonl(snap_dir: Path):
    pfile = snap_dir / "particles.jsonl"
    if not pfile.exists():
        return None, None
    pos, vel = [], []
    with open(pfile) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            p = json.loads(line)
            pos.append(p.get("pos", [0, 0, 0]))
            vel.append(p.get("vel", [0, 0, 0]))
    return np.array(pos), np.array(vel)

snap_final = out_cosmo / "snapshot_final"
if not snap_final.exists():
    snaps = sorted(out_cosmo.glob("snapshot*"))
    snap_final = snaps[-1] if snaps else None

pos_arr, vel_arr = (None, None)
if snap_final:
    pos_arr, vel_arr = load_particles_jsonl(snap_final)

if pos_arr is not None:
    # Diagrama de fase: posición X vs velocidad X
    ax5.scatter(pos_arr[:, 0], vel_arr[:, 0], s=1, alpha=0.4, c="tomato")
    ax5.set_xlabel("x [Mpc/h]")
    ax5.set_ylabel(r"$v_x$ [km/s]")
    ax5.set_title("Diagrama de fase (x, v_x)")
    ax5.grid(True, alpha=0.2)
else:
    ax5.text(0.5, 0.5, "Partículas no disponibles", ha="center", va="center",
             transform=ax5.transAxes, color="gray", fontsize=11)
    ax5.set_title("Diagrama de fase")

# ── Panel 6: Metadatos del run ─────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis("off")

checkpoint = out_cosmo / "snapshot_final" / "checkpoint.json"
if not checkpoint.exists():
    checkpoints = list(out_cosmo.glob("**/checkpoint.json"))
    checkpoint = checkpoints[-1] if checkpoints else None

meta_lines = ["Metadatos del run\n"]
if checkpoint and checkpoint.exists():
    with open(checkpoint) as f:
        ckpt = json.load(f)
    for k, v in ckpt.items():
        if isinstance(v, float):
            meta_lines.append(f"  {k}: {v:.6g}")
        else:
            meta_lines.append(f"  {k}: {v}")
else:
    meta_lines.append("  checkpoint.json no encontrado")

ax6.text(0.05, 0.95, "\n".join(meta_lines), transform=ax6.transAxes,
         verticalalignment="top", fontsize=9, fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
ax6.set_title("Metadatos")

plt.show()

## 4. Generar figuras de calidad publicación

El script `docs/scripts/generate_paper_figures.py` genera las figuras
usadas en el paper JOSS. Aquí lo ejecutamos directamente.

In [ ]:
paper_figs_script = PROJECT_ROOT / "docs" / "scripts" / "generate_paper_figures.py"

if paper_figs_script.exists():
    result = subprocess.run(
        [sys.executable, str(paper_figs_script), "--help"],
        capture_output=True,
        text=True,
    )
    print(result.stdout or result.stderr or "(sin help disponible)")
else:
    print(f"Script no encontrado: {paper_figs_script}")

## 5. Comparar con escala teórica

Comparamos el P(k) simulado con la aproximación de Harrison-Zel'dovich
P(k) ∝ k^{n_s} y con la supresión de transferencia de Eisenstein-Hu.

In [ ]:
pk_data = results.get("power_spectrum", results.get("pk", {}))
k_sim = safe_array(pk_data, "k", "k_h_mpc") if pk_data else None
pk_sim = safe_array(pk_data, "power", "pk_mpc3_h3") if pk_data else None

if k_sim is not None and pk_sim is not None:
    fig, ax = plt.subplots(figsize=(9, 6))

    mask = pk_sim > 0
    ax.loglog(k_sim[mask], pk_sim[mask], "o-", ms=5, lw=2,
              label="gadget-ng (N=512)", color="steelblue", zorder=3)

    # Pendiente de Harrison-Zel'dovich: n_s = 0.965
    k_th = k_sim[mask]
    n_s = 0.965
    pk_hz = k_th ** n_s
    pk_hz *= pk_sim[mask][len(pk_sim[mask]) // 4] / pk_hz[len(pk_hz) // 4]
    ax.loglog(k_th, pk_hz, "--", lw=1.5, alpha=0.7,
              label=rf"$P(k) \propto k^{{n_s}}$, $n_s={n_s}$", color="darkorange")

    ax.set_xlabel(r"$k$ [$h$/Mpc]", fontsize=13)
    ax.set_ylabel(r"$P(k)$ [Mpc/$h$]³", fontsize=13)
    ax.set_title("Espectro de potencia P(k) — ΛCDM N=512, z=0", fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, which="both")
    plt.tight_layout()
    plt.show()
else:
    print("P(k) no disponible. Ejecuta el Notebook 2 y genera los resultados de análisis.")

## Resumen

Con estos tres notebooks tienes un flujo de trabajo completo:

| Notebook | Contenido |
|----------|-----------|
| 01 | Primera simulación: Plummer, Kepler, visualización básica |
| 02 | Simulación ΛCDM: condiciones iniciales, integración cosmológica, P(k) |
| 03 | Herramientas de análisis: FoF, P(k), ξ(r), c(M), diagrama de fase |

Para ir más lejos:
- Activa MPI para cajas más grandes: `cargo build --release -p gadget-ng-cli --features mpi`
- Prueba SPH con `[sph]` en el TOML (gas + materia oscura)
- Mira los experimentos en `experiments/nbody/` para configuraciones avanzadas validadas